# Phase 5 (Colab) - Scalability test

**Question:** how does Spark's run time change when the data grows and when it gets more CPU cores?

We time **4 workloads** on **25%, 50% and 100%** of the clean review sample, first with **1 core**
(`local[1]`) and then with **every core Colab gives us** (`local[2]` on the free plan):

| Workload | Kind of work |
|---|---|
| sql_aggregation | Spark SQL group-by (reading data) |
| text_cleaning | Phase 2 cleaning + tokenizing (CPU heavy) |
| sentiment_scoring | the saved TF-IDF + Logistic Regression model on every review (ML inference) |
| als_training | implicit ALS, 5 iterations, on the 5-core ratings (iterative ML training) |

A small warm-up job runs before the timings so that Spark's start-up is not counted.
Results go to `MyDrive/amazon_reviews/results/scalability/scalability_runs.csv` (used by the web app).

Run **top to bottom**, CPU runtime. About **25-40 minutes**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
SAMPLE = '1pct'
LOCAL = '/content/scale'
OUT_DIR = f'{PROJECT}/results/scalability'
FRACTIONS = [0.25, 0.5, 1.0]
ALS_USER_SHARE = 0.05        # ALS uses 5% of the 5-core users (as in 04c), then 25/50/100% of those

import os
JAVA = '/usr/lib/jvm/java-17-openjdk-amd64'
if not os.path.exists(JAVA):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = JAVA
!pip -q install pyspark==3.5.3

!mkdir -p {LOCAL} {OUT_DIR}
!cp -r {PROJECT}/parquet/reviews_clean_{SAMPLE} {PROJECT}/models/sentiment_lr_{SAMPLE} {LOCAL}/
!cp {PROJECT}/raw/5core/Electronics.train.csv.gz {LOCAL}/

CORES = sorted({1, os.cpu_count()})
print('CPU cores on this Colab machine:', os.cpu_count(), '-> testing', CORES)

## Workloads (same rules as the rest of the project)

In [ ]:
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, StringIndexer
from pyspark.ml.recommendation import ALS

KEEP_WORDS = {'not', 'no', 'nor', 'against', 'very', 'too'}

def clean_text(title_col, text_col):
    c = F.concat_ws(' ', F.coalesce(title_col, F.lit('')), F.coalesce(text_col, F.lit('')))
    c = F.regexp_replace(c, r'<[^>]+>', ' ')
    c = F.regexp_replace(c, r'&#?[a-zA-Z0-9]+;', ' ')
    c = F.regexp_replace(c, r'https?://\S+|www\.\S+', ' ')
    c = F.lower(c)
    c = F.regexp_replace(c, r"n't\b", ' not')
    c = F.regexp_replace(c, r'[^a-z ]', ' ')
    c = F.regexp_replace(c, r'\s+', ' ')
    return F.trim(c)

def pick(df, frac):
    '''The same users every time, so each share contains the smaller ones.'''
    return df.filter(F.abs(F.hash('user_id')) % 1000 < int(frac * 1000))

def sql_aggregation(spark, frac):
    df = pick(spark.read.parquet(f'{LOCAL}/reviews_clean_{SAMPLE}'), frac)
    rows = df.groupBy('year', 'sentiment').agg(F.count('*').alias('n'), F.avg('word_count'), F.avg('helpful_vote')).collect()
    return sum(r['n'] for r in rows)

def text_cleaning(spark, frac):
    stop = [w for w in StopWordsRemover.loadDefaultStopWords('english') if w not in KEEP_WORDS]
    df = pick(spark.read.parquet(f'{LOCAL}/reviews_clean_{SAMPLE}'), frac).select('user_id', 'title', 'text')
    df = df.withColumn('text_clean', clean_text(F.col('title'), F.col('text')))
    df = RegexTokenizer(inputCol='text_clean', outputCol='t', pattern=r'\s+', minTokenLength=2).transform(df)
    df = StopWordsRemover(inputCol='t', outputCol='tokens', stopWords=stop).transform(df)
    row = df.agg(F.count('*').alias('n'), F.sum(F.size('tokens'))).first()   # forces all the work in one pass
    return row['n']

def sentiment_scoring(spark, frac):
    model = PipelineModel.load(f'{LOCAL}/sentiment_lr_{SAMPLE}')
    df = pick(spark.read.parquet(f'{LOCAL}/reviews_clean_{SAMPLE}'), frac).filter(F.size('tokens') > 0)
    row = model.transform(df).agg(F.count('*').alias('n'), F.sum('prediction')).first()
    return row['n']

def als_training(spark, frac):
    r = spark.read.csv(f'{LOCAL}/Electronics.train.csv.gz', header=True,
                       schema='user_id STRING, parent_asin STRING, rating DOUBLE, timestamp LONG')
    r = pick(r, ALS_USER_SHARE).filter(F.abs(F.hash('user_id', F.lit('als'))) % 1000 < int(frac * 1000))
    r = StringIndexer(inputCol='user_id', outputCol='u').fit(r).transform(r)
    r = StringIndexer(inputCol='parent_asin', outputCol='i').fit(r).transform(r).withColumn('bought', F.lit(1.0)).cache()
    n = r.count()
    ALS(userCol='u', itemCol='i', ratingCol='bought', implicitPrefs=True, alpha=10.0, rank=10,
        maxIter=5, regParam=0.1, seed=42).fit(r)
    r.unpersist()
    return n

WORKLOADS = [('sql_aggregation', sql_aggregation), ('text_cleaning', text_cleaning),
             ('sentiment_scoring', sentiment_scoring), ('als_training', als_training)]

## Run all timings (one fresh Spark session per core count)

In [ ]:
results = []
for cores in CORES:
    active = SparkSession.getActiveSession()
    if active:
        active.stop()
    spark = (SparkSession.builder.appName(f'scale-{cores}').master(f'local[{cores}]')
             .config('spark.driver.memory', '8g').config('spark.sql.shuffle.partitions', str(4 * cores))
             .getOrCreate())
    spark.sparkContext.setLogLevel('ERROR')
    spark.sparkContext.setCheckpointDir('/content/checkpoints')
    sql_aggregation(spark, 0.1)                     # warm-up: start-up cost is not timed
    for frac in FRACTIONS:
        for name, fn in WORKLOADS:
            t0 = time.time()
            rows = fn(spark, frac)
            sec = round(time.time() - t0, 1)
            results.append({'data_fraction': frac, 'cores': cores, 'step': name, 'seconds': sec, 'rows': rows,
                            'platform': f'Google Colab ({os.cpu_count()} vCPU), {SAMPLE} review sample'})
            print(f'{cores} core(s)  {int(frac * 100):>3}%  {name:<18} {sec:>7}s  rows={rows:,}', flush=True)

runs = pd.DataFrame(results)
runs.to_csv(f'{OUT_DIR}/scalability_runs.csv', index=False)
print('saved', f'{OUT_DIR}/scalability_runs.csv')

## Speed-up on 100% of the sample

In [ ]:
full = runs[runs['data_fraction'] == 1.0].pivot(index='step', columns='cores', values='seconds')
base = min(CORES)
for c in CORES:
    full[f'speed-up {c} cores'] = (full[base] / full[c]).round(2)
full

**How to read it:** speed-up = time with 1 core / time with more cores. The free Colab machine's
"2 cores" are two threads of one physical core, so expect less than 2x; reading files and the single
driver do not get faster (Amdahl's law). The Dataproc scripts repeat this study on 2, 4 and 6 real cores
over the full 43.9 M reviews.

## Download the result for the web app

In [ ]:
from google.colab import files
files.download(f'{OUT_DIR}/scalability_runs.csv')